In [1]:
from datasets import load_from_disk

DATASET_PATH = "/Volumes/Hardik/Arxiv_dataset/arxiv_50k_cleaned"

dataset = load_from_disk(DATASET_PATH)

train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

print(len(train))
print(len(validation))
print(len(test))

44971
2500
2500


In [2]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/hardikpathak/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/hardikpathak/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
import numpy as np
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.tokenize import sent_tokenize

In [16]:
def prepare_for_textrank(text):
    paragraphs = text.split(". ")
    cleaned = []

    for paragraph in paragraphs:
        if paragraph.count("&") >= 5:
            continue
        cleaned.append(paragraph)

    return ". ".join(cleaned)

In [17]:
sample = train[0]

text = prepare_for_textrank(sample["article"])

summary = textrank_summary(
    text,
    target_words=150
)

print("TEXT RANK SUMMARY:")
print(summary)

print("\n" + "="*80 + "\n")

print("REFERENCE ABSTRACT:")
print(sample["abstract"])

TEXT RANK SUMMARY:
comparing against radio observations for each nucleus, we vary magnetic field strength ( [MATH] ), wind speed ( [MATH] ), ionized gas density ( [MATH] ), and absorption fraction ( [MATH] ). combining each possible set of models from the eastern and western nucleus, we find that the resulting [MATH] -ray spectra peak around [MATH].3 gev with a maximum flux of [MATH] gev [MATH] s [MATH] ( see fig. in addition to accounting for [MATH] -ray and neutrino emission in arp 220, we have also take into account the effects of [MATH] [MATH] absorption due to the intense radiation fields in the nuclei. additionally, [MATH] [MATH] absorption of the tev energy [MATH] -rays make the tev [MATH] -ray flux a poor indicator of the neutrino flux in ulirgs and other such systems with extremely intense infrared radiation fields. princeton univ. princeton univ. press, princeton, nj ghisellini g., 2013, vol.


REFERENCE ABSTRACT:
the cores of arp 220, the closest ultraluminous infrared starb

In [18]:
from rouge_score import rouge_scorer

In [19]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

In [20]:
results = []

for i in range(100):
    article = prepare_for_textrank(test[i]["article"])
    reference = test[i]["abstract"]

    summary = textrank_summary(
        article,
        target_words=150
    )

    scores = scorer.score(reference, summary)

    results.append({
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure
    })

print("ROUGE-1:", np.mean([x["rouge1"] for x in results]))
print("ROUGE-2:", np.mean([x["rouge2"] for x in results]))
print("ROUGE-L:", np.mean([x["rougeL"] for x in results]))

ROUGE-1: 0.3372085062405113
ROUGE-2: 0.08557455377994491
ROUGE-L: 0.17943553784472752
